# 讓 Agent 先查資料再回答

這份教材先用一批小型參考資料示範資料如何進入流程、如何被查回來、如何影響最後回覆。正式文件搜尋可再接到語意搜尋設定。

In [ ]:
from pathlib import Path
import os, sys, subprocess

if not Path('agentic_sdk').exists():
    if not Path('Agentic-SDK').exists():
        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)
    os.chdir('Agentic-SDK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 準備參考資料

MVP 先用簡單資料表看懂流程。當資料變成 PDF、Markdown 或大量文字時，再換成語意搜尋。

In [ ]:
knowledge_items = [
    {'keywords': ['保存', 'bundle', '參考文件'], 'content': '使用參考文件的 Agent 需要保存 bundle，重新打開時才找得到原本的資料。'},
    {'keywords': ['runner', '唯讀'], 'content': '公開分享的 Runner 可以使用 Agent，但不能修改設定或保存。'},
    {'keywords': ['builder', '建立'], 'content': 'Builder 用來建立或調整 Agent 設定，再交給 Runner 試跑。'},
]

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    workflow_name='參考資料問答 Agent',
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=knowledge_items, fallback='目前沒有找到相關參考資料。'),
    action=DirectAnswerAction(),
)

result = workflow.run('為什麼使用參考文件的 Agent 要保存 bundle？')
print(result.final_message)
print('查回來的資料:', result.entities.get('latest_retrieved_content'))

## 換成正式文件搜尋時

如果資料不是幾筆固定條目，而是多份文件，就改用語意搜尋，並確認 `source-files`、`vectorstore` 與 bundle 都有被保存。